# 05 – Diffusion (ALR) Testing on North Carolina VIIRS DNB Data

This notebook demonstrates the **All-sky Light-pollution Ratio (ALR)** diffusion algorithm
originally implemented by [Katy Abbott for Big Bend National Park](https://github.com/BigBendNP/nightskyquality),
ported here to work with `scipy`, `rioxarray`, and `geopandas`.

## What is ALR?
ALR quantifies how much artificial sky glow a location receives, accounting for light sources
up to **300 km away**.  It is calculated by convolving the radiance image with a series of
annular kernels, each weighted by the Duriscoe distance function:

$$\alpha(d) = 2.3 \left(\frac{d}{350}\right)^{0.28}$$

$$\text{ALR} = \frac{1}{562.72} \sum_{i=1}^{N} d_i^{-\alpha(d_i)} \cdot (r \ast k_i)$$

where $k_i$ is the annular kernel for ring $i$, $r$ is the radiance image, and $d_i$ is the
average radius of ring $i$ in km.

| Sky quality class | ALR |
|---|---|
| Good | 0 – 0.33 |
| Moderate (threatened) | 0.33 – 2.0 |
| Poor (sensitive protected areas) | 2.0 – 10.0 |
| Milky Way invisible | > 10.0 |

---
**Reference:** Duriscoe, D. M., Anderson, C. B., Duriscoe, E. M., & Eck, K. M. (2018).
*A simplified model of all-sky artificial sky glow derived from VIIRS Day/Night Band data.*
Journal of Quantitative Spectroscopy and Radiative Transfer, 212, 1–8.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import rioxarray

import config
from diffusion import calculate_alr_for_region, calculate_alr, circular_annulus_footprint

## 1. Load VIIRS DNB Radiance Data

In [ ]:
viirs_da = rioxarray.open_rasterio(config.RADIANCE_OLDER, masked=True)
print("VIIRS CRS :", viirs_da.rio.crs)
print("VIIRS shape:", viirs_da.shape)
print("VIIRS res  :", viirs_da.rio.resolution())
viirs_da

## 2. Load North Carolina Boundary (from TIGER shapefile)

In [ ]:
states = gpd.read_file(config.STATES_SHP)
nc = states[states["NAME"] == "North Carolina"].copy()
print(nc[["NAME", "STUSPS", "geometry"]])
nc.plot()
plt.title("North Carolina boundary")
plt.show()

## 3. Visualise Annular Kernels

The algorithm builds a separate kernel for each distance ring, then
convolves the radiance image with each kernel.

In [ ]:
pixel_size_km = 0.45  # 450 m target resolution
max_radius_km = 300.0
n_rings = 38

annulus_cells = np.linspace(0, max_radius_km / pixel_size_km, n_rings + 1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
sample_indices = np.linspace(0, n_rings - 1, 10, dtype=int)

for ax, idx in zip(axes.ravel(), sample_indices):
    eps = 0.001 if idx == 0 else 0
    r_inner = annulus_cells[idx] + eps
    r_outer = annulus_cells[idx + 1]
    k = circular_annulus_footprint(r_inner, r_outer)
    radius_km = 0.5 * (r_inner + r_outer) * pixel_size_km
    ax.imshow(k, cmap="Blues", interpolation="nearest")
    ax.set_title(f"Ring {idx}\n≈{radius_km:.0f} km", fontsize=8)
    ax.axis("off")

fig.suptitle("Sample annular convolution kernels", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Visualise the Distance-Weighting Function

Each ring's contribution is down-weighted by $d^{-\alpha(d)}$ where
$\alpha(d) = 2.3 (d/350)^{0.28}$.  More distant sources are discounted more steeply.

In [ ]:
def alpha(d_km):
    return 2.3 * (d_km / 350.0) ** 0.28

d_range = np.linspace(0.5, 300, 500)
weights = d_range ** (-alpha(d_range))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(d_range, alpha(d_range))
axes[0].set_xlabel("Distance (km)")
axes[0].set_ylabel("α(d)")
axes[0].set_title("Duriscoe exponent α(d)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(d_range, weights)
axes[1].set_xlabel("Distance (km)")
axes[1].set_ylabel(r"$d^{-\alpha(d)}$")
axes[1].set_title("Distance weight applied to each ring")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Run ALR on North Carolina (with 300 km Buffer)

The function:
1. Reprojects NC to EPSG:5070 (Conus Albers Equal Area – a metric equal-area CRS for CONUS)
2. Creates a 300 km buffer
3. Clips and reprojects the VIIRS tile to that buffer at 450 m resolution
4. Computes ALR via annular FFT convolutions
5. Clips the result back to NC's boundary

> ⚠️  This cell may take several minutes to run.

In [ ]:
import time

# EPSG:5070 = Conus Albers Equal Area (metres) – appropriate for CONUS regions
ALBERS_EPSG = 5070

t0 = time.time()
alr_nc = calculate_alr_for_region(
    viirs_da=viirs_da,
    region_gdf=nc,
    albers_epsg=ALBERS_EPSG,
    target_res_m=450.0,
    n_rings=38,
)
elapsed = time.time() - t0
print(f"ALR calculation complete in {elapsed:.1f} s")
print("ALR shape:", alr_nc.shape)
print("ALR min/max:", float(np.nanmin(alr_nc.values)), "/", float(np.nanmax(alr_nc.values)))

## 6. Visualise Observed Radiance (VIIRS)

In [ ]:
# Clip the raw radiance to NC for comparison
nc_albers = nc.to_crs(epsg=ALBERS_EPSG)
viirs_albers = viirs_da.rio.reproject(f"EPSG:{ALBERS_EPSG}", resolution=450.0, nodata=np.nan)
viirs_nc = viirs_albers.rio.clip(nc_albers.geometry, all_touched=True, drop=True)

rad_arr = viirs_nc.values.squeeze()
rad_arr[rad_arr < 0] = np.nan

fig, ax = plt.subplots(figsize=(10, 6))
img = ax.imshow(
    rad_arr,
    cmap="inferno",
    norm=mcolors.LogNorm(vmin=0.5, vmax=np.nanpercentile(rad_arr, 99)),
    origin="upper",
)
plt.colorbar(img, ax=ax, label="Radiance (nW·cm⁻²·sr⁻¹)", shrink=0.7)
ax.set_title("VIIRS DNB Observed Radiance – North Carolina", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Visualise ALR Result

In [ ]:
alr_arr = alr_nc.values.squeeze()

fig, ax = plt.subplots(figsize=(10, 6))
img = ax.imshow(
    alr_arr,
    cmap="hot_r",
    norm=mcolors.LogNorm(vmin=0.1, vmax=np.nanpercentile(alr_arr[alr_arr > 0], 99)),
    origin="upper",
)
plt.colorbar(img, ax=ax, label="ALR (unitless)", shrink=0.7)
ax.set_title("All-sky Light Pollution Ratio (ALR) – North Carolina", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Sky Quality Classification

In [ ]:
# Classify into four sky-quality bands per Duriscoe et al.
bins = [0, 0.33, 2.0, 10.0, np.inf]
labels = ["Good (0–0.33)", "Moderate (0.33–2)", "Poor (2–10)", "Milky Way invisible (>10)"]
colors = ["#1a9641", "#fdae61", "#d7191c", "#7b2d8b"]

classified = np.digitize(alr_arr, bins) - 1  # 0-indexed
classified = classified.astype(float)
classified[np.isnan(alr_arr)] = np.nan

cmap = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm(range(len(bins)), cmap.N)

fig, ax = plt.subplots(figsize=(10, 6))
img = ax.imshow(classified, cmap=cmap, norm=norm, origin="upper", interpolation="nearest")
cbar = plt.colorbar(img, ax=ax, shrink=0.7, ticks=[0.5, 1.5, 2.5, 3.5])
cbar.ax.set_yticklabels(labels, fontsize=8)
ax.set_title("Night Sky Quality Classification – North Carolina", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Summary Statistics

In [ ]:
import pandas as pd

valid = alr_arr[~np.isnan(alr_arr) & (alr_arr > 0)]

rows = []
for low, high, label in zip(bins[:-1], bins[1:], labels):
    mask = (valid >= low) & (valid < high)
    pct = 100.0 * mask.sum() / len(valid)
    rows.append({"Sky quality class": label, "ALR range": f"{low}–{high}", "% of NC pixels": f"{pct:.1f}%"})

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f"\nMedian ALR : {np.median(valid):.3f}")
print(f"Mean ALR   : {np.mean(valid):.3f}")
print(f"Max ALR    : {np.max(valid):.3f}")

## 10. Side-by-Side Comparison: Radiance vs. ALR

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(
    rad_arr,
    cmap="inferno",
    norm=mcolors.LogNorm(vmin=0.5, vmax=np.nanpercentile(rad_arr, 99)),
    origin="upper",
)
axes[0].set_title("Observed Radiance (VIIRS DNB)", fontsize=11)
axes[0].axis("off")

im = axes[1].imshow(
    alr_arr,
    cmap="hot_r",
    norm=mcolors.LogNorm(vmin=0.1, vmax=np.nanpercentile(alr_arr[alr_arr > 0], 99)),
    origin="upper",
)
axes[1].set_title("All-sky Light Pollution Ratio (ALR)", fontsize=11)
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], label="ALR", shrink=0.7)

fig.suptitle("North Carolina – Radiance vs. ALR", fontsize=13)
plt.tight_layout()
plt.show()